# 06 - Advanced PDF Parsing & Text Normalization

> Complete guide and in-depth Arabic documentation: [`README.md`](./README.md)

Handles PDF formatting anomalies: de-hyphenation, ligatures, Arabic normalization, and header stripping.

## 1. Load Raw Sample PDF

In [ ]:
def clean_text(text: str) -> str:
    # 1. استبدال الـ Ligatures الشائعة
    text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
    # 2. توحيد المسافات والأسطر الفارغة المكررة
    text = " ".join(text.split())
    return text.strip()

# نص تجريبي مشوه كما يخرج أحياناً من محركات استخراج الـ PDF
raw_pdf_text = """Company Financial Report 2024\n\n\n
The financial   performance for   fiscal year 2024 shows significant   growth in proﬁtability.\n\n
Revenue increased by 25% due to efﬁcient operations."""

cleaned = clean_text(raw_pdf_text)

print("--- النص الخام قبل التنظيف ---")
print(repr(raw_pdf_text))

print("\n--- النص بعد التنظيف ---")
print(cleaned)

## 2. De-Hyphenation & Broken Line Fixes
Merges hyphenated words broken across line wraps (e.g. 'infor-\nmation' -> 'information').

In [ ]:
import os
from typing import List
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

class SmartPDFProcessor:
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", " ", ""]
        )

    def _clean_text(self, text: str) -> str:
        # استبدال الحروف المشفرة وتنظيف المسافات
        text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
        return " ".join(text.split()).strip()

    def process_pdf(self, pdf_path: str) -> List[Document]:
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"الملف غير موجود: {pdf_path}")

        loader = PyPDFLoader(pdf_path)
        pages = loader.load()
        total_pages = len(pages)
        processed_chunks: List[Document] = []

        for page_num, page in enumerate(pages, 1):
            clean_content = self._clean_text(page.page_content)

            # تخطي الصفحات شبه الفارغة (أقل من 50 حرف)
            if len(clean_content) < 50:
                print(f"⚠️ تم تخطي الصفحة {page_num} لقلة المحتوى ({len(clean_content)} حرف).")
                continue

            # إثراء الميتا-داتا
            enhanced_metadata = {
                **page.metadata,
                "page_number": page_num,
                "total_pages": total_pages,
                "chunk_method": "smart_recursive",
                "char_count": len(clean_content)
            }

            # إنشاء قطع Document مجزأة مع الميتا-داتا الجديدة
            chunks = self.text_splitter.create_documents(
                texts=[clean_content],
                metadatas=[enhanced_metadata]
            )
            processed_chunks.extend(chunks)

        return processed_chunks

print("✅ تم تعريف كلاس SmartPDFProcessor بنجاح!")

## 3. Ligatures & Arabic Text Normalization
Normalizes character forms, strips tashkeel, and replaces typographic ligatures (fi, fl).

In [ ]:
# اختبار الكلاس الذكي على التقرير المالي التجريبي
from pathlib import Path
data_dir = Path('data') if Path('data').exists() else Path('../../data')
pdf_file = str((data_dir / 'pdf' / 'sample_financial_report.pdf').resolve())

processor = SmartPDFProcessor(chunk_size=300, chunk_overlap=40)
chunks = processor.process_pdf(pdf_file)

print(f"[+] إجمالي القطع الناتجة بعد التنظيف والفلترة: {len(chunks)}\n")
for i, chunk in enumerate(chunks, 1):
    print(f"--- القطعة #{i} (صفحة {chunk.metadata['page_number']}) ---")
    print(chunk.page_content[:180] + '...')
    print(f"الميتا-داتا: {chunk.metadata}\n")


## 4. Header & Footer Detection and Stripping

In [ ]:
for i, chunk in enumerate(smart_chunks, 1):
    print(f"=== Chunk {i} ===")
    print(f"المحتوى: {chunk.page_content}")
    print("\nالبيانات الوصفية المثرية (Enriched Metadata):")
    for k, v in chunk.metadata.items():
        print(f"  • {k}: {v}")
    print("=" * 50)

## 5. End-to-End Smart PDF Pipeline with SHA-256 Deduplication